# AutoDS Agent – Demo Notebook

End-to-end autonomous data science on any tabular dataset.

**Workflow**
```
Dataset → DataAgent → EDAPipeline → MLAgent → InsightAgent → Report
```

In [ ]:
import sys
sys.path.insert(0, '../..')          # add project root to path
import pprint
from IPython.display import Markdown, display
from pipelines.eda_pipeline import EDAPipeline
from pipelines.training_pipeline import TrainingPipeline

## 1 · Load a sample dataset (Titanic via URL)

In [2]:
import pandas as pd

url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df  = pd.read_csv(url)
df.to_csv('../data/titanic.csv', index=False)
print(df.shape)
df.head()

## 2 · Run EDA Pipeline

In [3]:
eda    = EDAPipeline()
result, brief, report = eda.run(data_path='../data/titanic.csv', target_hint='Survived', report_title='Titanic EDA Report')
print(result)

In [5]:
print('\nProfile summary:')
from agents.data_agent import DataAgent
da = DataAgent(data_path='../data/titanic.csv', target_feature='Survived')
da.raw_df = result['clean_df']   # reuse already-loaded df
da.profile = result['profile']
print(da.get_profile_summary())

## 3 · View generated charts

In [8]:
from IPython.display import Image, display
for plot in report['plots']:
    display(Image(plot))

## 4 · Run Training Pipeline

In [9]:
trainer = TrainingPipeline(target_feature='Survived')

In [14]:
train_result = trainer.run(
    source='../data/titanic.csv',
    report_title='Titanic Survival Prediction'
    )

In [34]:
print(train_result.brief['llm_summary'])

## 5 · Inspect Metrics & Insights

In [ ]:
def printmd(content):
    if isinstance(content, list):
        content = "\n\n".join(content)  # extra spacing between bullets/sections
    display(Markdown(content))

In [46]:
def to_markdown(train_result):
    md = []

    # EDA
    md.append("## EDA Insights")
    md.append(train_result.brief['llm_summary'])

    # Model performance
    md.append("## Model Performance")
    md.append(f"**Best model:** {train_result.ml_result['best_model_name']}")
    md.append(f"**Task type:** {train_result.ml_result['task_type']}")

    metrics = {
        k: v for k, v in train_result.ml_result['metrics'].items()
        if k != 'classification_report'
    }

    md.append("**Metrics:**")
    for k, v in metrics.items():
        md.append(f"- **{k}**: {v}")

    # Recommendations
    md.append("## Recommendations")
    recs = train_result.report['recommendations']
    if isinstance(recs, list):
        md.extend(recs)
    else:
        md.append(recs)

    # Next steps
    md.append("## Next Steps")
    steps = train_result.report['next_steps']
    if isinstance(steps, list):
        md.extend(steps)
    else:
        md.append(steps)

    return "\n\n".join(md)

In [47]:
printmd(to_markdown(train_result))